# RAG Expert to answer user question based on document knowledge base
Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [ ]:
import sys
from pathlib import Path
from pydantic import BaseModel, Field

In [ ]:

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root (.git) from current location")


start_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
repo_root = find_repo_root(start_dir)
module_dir = repo_root / "week5" / "community-contributions" / "ParulLlm"

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from pdf_to_md_converter import convert_pdfs_to_markdown

pdf_dir = module_dir / "EPMDocs" / "pdfs"
out_dir = module_dir / "EPMDocs" / "markdown"

count = convert_pdfs_to_markdown(pdf_dir, out_dir)
print(f"Converted {count} PDF(s) to Markdown in {out_dir.relative_to(repo_root)}")

In [ ]:
def fetch_documents(knowledge_base_path=None):
    """A homemade version of the LangChain DirectoryLoader."""

    if knowledge_base_path is None:
        knowledge_base_path = module_dir / "EPMDocs" / "markdown"

    documents = []

    for file in knowledge_base_path.rglob("*.md"):
        doc_type = file.parent.name
        with open(file, "r", encoding="utf-8") as f:
            documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents from {knowledge_base_path}")
    return documents

In [ ]:
documents = fetch_documents()
print(documents)

In [ ]:
# A class to perfectly represent a chunk

class Result(BaseModel):
    page_content: str
    metadata: dict

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]


In [ ]:
def make_chunks_prompt(document):
    chunk_size = (len(document["text"]) // 300) + 1
    return f"""
    You take a document and you split the document into overlapping chunks for a KnowledgeBase.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}
The document is from travel company Expedia and having steps about transferring financial and planning data from Oracle EPM 
Cloud Planning to Expedia Group Data Lake (EGDL) . Also Programmatic access user guide.

A chatbot will use these chunks to answer questions about running these data transfer jobs and 
how to get the Programmatic access .
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {chunk_size} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
    """ 
    

In [ ]:
print(make_chunks_prompt(documents[0]))

In [ ]:
def built_msg(document):
 return [
        {"role": "user", "content": make_chunks_prompt(document)},
    ]

In [ ]:
from openai import OpenAI
MODEL = "gpt-4.1-nano"
openai = OpenAI()
from litellm import completion
def process_document(document):
    messages = built_msg(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    # print("reply" + reply)
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [ ]:
process_document(documents[0])

In [ ]:
from tqdm import tqdm
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks
chunks= create_chunks(documents)
print(len(chunks))

In [ ]:
from chromadb import PersistentClient

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [ ]:
create_embeddings(chunks)

In [ ]:
import numpy as np
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
print(doc_types)
colors = [['blue', 'green'][['markdown', 'markdown'].index(t)] for t in doc_types]

In [ ]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go

n_samples = len(vectors)
if n_samples < 2:
    raise ValueError("Need at least 2 samples to run t-SNE")

# sklearn requires perplexity < n_samples
perplexity = min(30, n_samples - 1)
tsne = TSNE(n_components=2, random_state=20, perplexity=perplexity)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title=f'2D Chroma Vector Store Visualization (perplexity={perplexity})',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [ ]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [ ]:
question = "How to run active forecast"
chunks = fetch_context_unranked(question)
for chunk in chunks:
    print(chunk.page_content[:35]+"...")

In [ ]:
reranked= rerank(question, chunks)
for chunk in reranked:
    print(chunk.page_content[:35]+"...")

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company's planning and forecatsing data tables and access.
You are chatting with a user about how to access different data tables and run different airflow jobs.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company's planning and forecatsing data tables and access.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
rewrite_query("How to run active forecast", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    print(history)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("How to run active forecast", [])

In [ ]:
History = answer_question("what parameters to be passed to run active forecast", [])

In [ ]:
answer_question("what parameters to be passed to run Plan & Archived forecast", [])

In [ ]:
answer_question("how can i access the b2b brands", [])